<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day6_5_(260615)_CRUD_%EA%B8%B0%EB%B0%98_Thymeleaf_%EC%9B%B9_%EC%95%A0%ED%94%8C%EB%A6%AC%EC%BC%80%EC%9D%B4%EC%85%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%writefile ~/springBoot/security-jpa-board/build.gradle

plugins {
    id 'java'
    id 'org.springframework.boot' version '3.3.5'
    id 'io.spring.dependency-management' version '1.1.6'
}

group = 'com.example'
version = '0.0.1-SNAPSHOT'

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(21)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-thymeleaf'
    implementation 'org.springframework.boot:spring-boot-starter-security'
    implementation 'org.springframework.boot:spring-boot-starter-data-jpa'
    implementation 'org.springframework.boot:spring-boot-starter-validation'
    runtimeOnly 'org.mariadb.jdbc:mariadb-java-client'

    testImplementation 'org.springframework.boot:spring-boot-starter-test'
    testImplementation 'org.springframework.security:spring-security-test'
    testRuntimeOnly 'org.junit.platform:junit-platform-launcher'
}

tasks.named('test') {
    useJUnitPlatform()
}

Overwriting /root/springBoot/security-jpa-board/build.gradle


In [5]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/application.properties

server.port=3001

spring.application.name=security-jpa-board

spring.datasource.url=jdbc:mariadb://localhost:3306/testdb
spring.datasource.username=testuser
spring.datasource.password=1234
spring.datasource.driver-class-name=org.mariadb.jdbc.Driver

spring.jpa.hibernate.ddl-auto=update
spring.jpa.show-sql=true
spring.jpa.properties.hibernate.format_sql=true

spring.thymeleaf.cache=false

logging.level.org.hibernate.SQL=debug
logging.level.org.hibernate.orm.jdbc.bind=trace

Overwriting /root/springBoot/security-jpa-board/src/main/resources/application.properties


In [6]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/domain/Role.java

package com.example.board.domain;

public enum Role {
    USER,
    ADMIN
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/domain/Role.java


In [7]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/domain/User.java

package com.example.board.domain;

import jakarta.persistence.*;
import java.time.LocalDateTime;

@Entity
@Table(name = "users")
public class User {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(nullable = false, unique = true, length = 50)
    private String username;

    @Column(nullable = false)
    private String password;

    @Column(nullable = false, length = 30)
    private String displayName;

    @Enumerated(EnumType.STRING)
    @Column(nullable = false, length = 20)
    private Role role;

    @Column(nullable = false)
    private LocalDateTime createdAt;

    protected User() {
    }

    public User(String username, String password, String displayName, Role role) {
        this.username = username;
        this.password = password;
        this.displayName = displayName;
        this.role = role;
        this.createdAt = LocalDateTime.now();
    }

    public Long getId() {
        return id;
    }

    public String getUsername() {
        return username;
    }

    public String getPassword() {
        return password;
    }

    public String getDisplayName() {
        return displayName;
    }

    public Role getRole() {
        return role;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/domain/User.java


In [8]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/domain/Post.java

package com.example.board.domain;

import jakarta.persistence.*;
import java.time.LocalDateTime;

@Entity
@Table(name = "posts")
public class Post {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(nullable = false, length = 200)
    private String title;

    @Lob
    @Column(nullable = false)
    private String content;

    @ManyToOne(fetch = FetchType.LAZY)
    @JoinColumn(name = "user_id", nullable = false)
    private User writer;

    @Column(nullable = false)
    private LocalDateTime createdAt;

    private LocalDateTime updatedAt;

    protected Post() {
    }

    public Post(String title, String content, User writer) {
        this.title = title;
        this.content = content;
        this.writer = writer;
        this.createdAt = LocalDateTime.now();
    }

    public void update(String title, String content) {
        this.title = title;
        this.content = content;
        this.updatedAt = LocalDateTime.now();
    }

    public boolean isWrittenBy(String username) {
        return this.writer.getUsername().equals(username);
    }

    public Long getId() {
        return id;
    }

    public String getTitle() {
        return title;
    }

    public String getContent() {
        return content;
    }

    public User getWriter() {
        return writer;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }

    public LocalDateTime getUpdatedAt() {
        return updatedAt;
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/domain/Post.java


In [9]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/dto/SignupRequest.java

package com.example.board.dto;

import com.example.board.domain.Role;
import jakarta.validation.constraints.NotBlank;
import jakarta.validation.constraints.NotNull;
import jakarta.validation.constraints.Size;

public class SignupRequest {

    @NotBlank(message = "아이디를 입력하세요.")
    @Size(min = 4, max = 50, message = "아이디는 4자 이상 50자 이하로 입력하세요.")
    private String username;

    @NotBlank(message = "비밀번호를 입력하세요.")
    @Size(min = 4, max = 100, message = "비밀번호는 4자 이상 입력하세요.")
    private String password;

    @NotBlank(message = "이름을 입력하세요.")
    @Size(max = 30, message = "이름은 30자 이하로 입력하세요.")
    private String displayName;

    @NotNull(message = "권한을 선택하세요.")
    private Role role = Role.USER;

    public String getUsername() {
        return username;
    }

    public void setUsername(String username) {
        this.username = username;
    }

    public String getPassword() {
        return password;
    }

    public void setPassword(String password) {
        this.password = password;
    }

    public String getDisplayName() {
        return displayName;
    }

    public void setDisplayName(String displayName) {
        this.displayName = displayName;
    }

    public Role getRole() {
        return role;
    }

    public void setRole(Role role) {
        this.role = role;
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/dto/SignupRequest.java


In [10]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/dto/PostRequest.java

package com.example.board.dto;

import jakarta.validation.constraints.NotBlank;
import jakarta.validation.constraints.Size;

public class PostRequest {

    @NotBlank(message = "제목을 입력하세요.")
    @Size(max = 200, message = "제목은 200자 이하로 입력하세요.")
    private String title;

    @NotBlank(message = "내용을 입력하세요.")
    private String content;

    public PostRequest() {
    }

    public PostRequest(String title, String content) {
        this.title = title;
        this.content = content;
    }

    public String getTitle() {
        return title;
    }

    public String getContent() {
        return content;
    }

    public void setTitle(String title) {
        this.title = title;
    }

    public void setContent(String content) {
        this.content = content;
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/dto/PostRequest.java


In [11]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/repository/UserRepository.java

package com.example.board.repository;

import com.example.board.domain.User;
import org.springframework.data.jpa.repository.JpaRepository;

import java.util.Optional;

public interface UserRepository extends JpaRepository<User, Long> {

    Optional<User> findByUsername(String username);

    boolean existsByUsername(String username);
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/repository/UserRepository.java


In [12]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/repository/PostRepository.java

package com.example.board.repository;

import com.example.board.domain.Post;
import org.springframework.data.jpa.repository.JpaRepository;

import java.util.List;

public interface PostRepository extends JpaRepository<Post, Long> {

    List<Post> findAllByOrderByIdDesc();
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/repository/PostRepository.java


In [13]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/service/AuthService.java

package com.example.board.service;

import com.example.board.domain.User;
import com.example.board.dto.SignupRequest;
import com.example.board.repository.UserRepository;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

@Service
public class AuthService {

    private final UserRepository userRepository;
    private final PasswordEncoder passwordEncoder;

    public AuthService(UserRepository userRepository,
                       PasswordEncoder passwordEncoder) {
        this.userRepository = userRepository;
        this.passwordEncoder = passwordEncoder;
    }

    @Transactional
    public void signup(SignupRequest request) {
        if (userRepository.existsByUsername(request.getUsername())) {
            throw new IllegalArgumentException("이미 사용 중인 아이디입니다.");
        }

        String encodedPassword = passwordEncoder.encode(request.getPassword());

        User user = new User(
                request.getUsername(),
                encodedPassword,
                request.getDisplayName(),
                request.getRole()
        );

        userRepository.save(user);
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/service/AuthService.java


In [14]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/service/CustomUserDetailsService.java

package com.example.board.service;

import com.example.board.domain.User;
import com.example.board.repository.UserRepository;
import org.springframework.security.core.authority.SimpleGrantedAuthority;
import org.springframework.security.core.userdetails.UserDetails;
import org.springframework.security.core.userdetails.UserDetailsService;
import org.springframework.security.core.userdetails.UsernameNotFoundException;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

import java.util.List;

@Service
public class CustomUserDetailsService implements UserDetailsService {

    private final UserRepository userRepository;

    public CustomUserDetailsService(UserRepository userRepository) {
        this.userRepository = userRepository;
    }

    @Override
    @Transactional(readOnly = true)
    public UserDetails loadUserByUsername(String username) throws UsernameNotFoundException {
        User user = userRepository.findByUsername(username)
                .orElseThrow(() -> new UsernameNotFoundException("회원을 찾을 수 없습니다."));

        return new org.springframework.security.core.userdetails.User(
                user.getUsername(),
                user.getPassword(),
                List.of(new SimpleGrantedAuthority("ROLE_" + user.getRole().name()))
        );
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/service/CustomUserDetailsService.java


In [15]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/config/SecurityConfig.java

package com.example.board.config;

import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.security.config.annotation.web.builders.HttpSecurity;
import org.springframework.security.config.annotation.web.configuration.EnableWebSecurity;
import org.springframework.security.crypto.bcrypt.BCryptPasswordEncoder;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.security.web.SecurityFilterChain;

@Configuration
@EnableWebSecurity
public class SecurityConfig {

    @Bean
    public SecurityFilterChain securityFilterChain(HttpSecurity http) throws Exception {
        http
                .authorizeHttpRequests(auth -> auth
                        .requestMatchers("/", "/auth/signup", "/login", "/css/**").permitAll()
                        .requestMatchers("/posts/new").hasAnyRole("USER", "ADMIN")
                        .requestMatchers("/posts/*/edit", "/posts/*/delete").hasAnyRole("USER", "ADMIN")
                        .requestMatchers("/posts/**").permitAll()
                        .anyRequest().authenticated()
                )
                .formLogin(form -> form
                        .loginPage("/login")
                        .defaultSuccessUrl("/posts", true)
                        .permitAll()
                )
                .logout(logout -> logout
                        .logoutUrl("/logout")
                        .logoutSuccessUrl("/posts")
                        .invalidateHttpSession(true)
                        .deleteCookies("JSESSIONID")
                );

        return http.build();
    }

    @Bean
    public PasswordEncoder passwordEncoder() {
        return new BCryptPasswordEncoder();
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/config/SecurityConfig.java


In [16]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/service/PostService.java

package com.example.board.service;

import com.example.board.domain.Post;
import com.example.board.domain.Role;
import com.example.board.domain.User;
import com.example.board.dto.PostRequest;
import com.example.board.repository.PostRepository;
import com.example.board.repository.UserRepository;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

import java.util.List;

@Service
public class PostService {

    private final PostRepository postRepository;
    private final UserRepository userRepository;

    public PostService(PostRepository postRepository,
                       UserRepository userRepository) {
        this.postRepository = postRepository;
        this.userRepository = userRepository;
    }

    @Transactional(readOnly = true)
    public List<Post> findAll() {
        return postRepository.findAllByOrderByIdDesc();
    }

    @Transactional(readOnly = true)
    public Post findById(Long id) {
        return postRepository.findById(id)
                .orElseThrow(() -> new IllegalArgumentException("게시글을 찾을 수 없습니다."));
    }

    @Transactional
    public Long create(PostRequest request, String username) {
        User writer = userRepository.findByUsername(username)
                .orElseThrow(() -> new IllegalArgumentException("작성자를 찾을 수 없습니다."));

        Post post = new Post(
                request.getTitle(),
                request.getContent(),
                writer
        );

        Post savedPost = postRepository.save(post);

        return savedPost.getId();
    }

    @Transactional
    public void update(Long id, PostRequest request, String username) {
        Post post = findById(id);

        if (!canManage(post, username)) {
            throw new IllegalArgumentException("작성자 또는 관리자만 수정할 수 있습니다.");
        }

        post.update(request.getTitle(), request.getContent());
    }

    @Transactional
    public void delete(Long id, String username) {
        Post post = findById(id);

        if (!canManage(post, username)) {
            throw new IllegalArgumentException("작성자 또는 관리자만 삭제할 수 있습니다.");
        }

        postRepository.delete(post);
    }

    @Transactional(readOnly = true)
    public boolean canManage(Post post, String username) {
        User loginUser = userRepository.findByUsername(username)
                .orElseThrow(() -> new IllegalArgumentException("로그인 사용자를 찾을 수 없습니다."));

        return post.isWrittenBy(username) || loginUser.getRole() == Role.ADMIN;
    }
}


Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/service/PostService.java


In [17]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/controller/HomeController.java

package com.example.board.controller;

import org.springframework.stereotype.Controller;
import org.springframework.web.bind.annotation.GetMapping;

@Controller
public class HomeController {

    @GetMapping("/")
    public String home() {
        return "redirect:/posts";
    }

    @GetMapping("/login")
    public String login() {
        return "auth/login";
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/controller/HomeController.java


In [18]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/controller/AuthController.java

package com.example.board.controller;

import com.example.board.dto.SignupRequest;
import com.example.board.service.AuthService;
import jakarta.validation.Valid;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.validation.BindingResult;
import org.springframework.web.bind.annotation.*;

@Controller
@RequestMapping("/auth")
public class AuthController {

    private final AuthService authService;

    public AuthController(AuthService authService) {
        this.authService = authService;
    }

    @GetMapping("/signup")
    public String signupForm(Model model) {
        model.addAttribute("signupRequest", new SignupRequest());
        return "auth/signup";
    }

    @PostMapping("/signup")
    public String signup(@Valid @ModelAttribute SignupRequest signupRequest,
                         BindingResult bindingResult,
                         Model model) {
        if (bindingResult.hasErrors()) {
            return "auth/signup";
        }

        try {
            authService.signup(signupRequest);
        } catch (IllegalArgumentException e) {
            model.addAttribute("signupError", e.getMessage());
            return "auth/signup";
        }

        return "redirect:/login";
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/controller/AuthController.java


In [19]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/controller/PostController.java

package com.example.board.controller;

import com.example.board.domain.Post;
import com.example.board.dto.PostRequest;
import com.example.board.service.PostService;
import jakarta.validation.Valid;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.validation.BindingResult;
import org.springframework.web.bind.annotation.*;

import java.util.List;

@Controller
@RequestMapping("/posts")
public class PostController {

    private final PostService postService;

    public PostController(PostService postService) {
        this.postService = postService;
    }

    @GetMapping
    public String list(Model model, Authentication authentication) {
        List<Post> posts = postService.findAll();

        model.addAttribute("posts", posts);
        model.addAttribute("loginUsername", getLoginUsername(authentication));
        model.addAttribute("isAdmin", isAdmin(authentication));

        return "posts/list";
    }

    @GetMapping("/{id}")
    public String detail(@PathVariable Long id,
                         Model model,
                         Authentication authentication) {
        Post post = postService.findById(id);
        String loginUsername = getLoginUsername(authentication);

        model.addAttribute("post", post);
        model.addAttribute("loginUsername", loginUsername);
        model.addAttribute("isAdmin", isAdmin(authentication));
        model.addAttribute("canManage", canManage(post, loginUsername, authentication));

        return "posts/detail";
    }

    @GetMapping("/new")
    public String createForm(Model model) {
        model.addAttribute("postRequest", new PostRequest());
        return "posts/form";
    }

    @PostMapping
    public String create(@Valid @ModelAttribute PostRequest postRequest,
                         BindingResult bindingResult,
                         Authentication authentication) {
        if (bindingResult.hasErrors()) {
            return "posts/form";
        }

        String username = authentication.getName();
        Long postId = postService.create(postRequest, username);

        return "redirect:/posts/" + postId;
    }

    @GetMapping("/{id}/edit")
    public String editForm(@PathVariable Long id,
                           Model model,
                           Authentication authentication) {
        Post post = postService.findById(id);
        String username = authentication.getName();

        if (!postService.canManage(post, username)) {
            throw new IllegalArgumentException("작성자 또는 관리자만 수정할 수 있습니다.");
        }

        PostRequest postRequest = new PostRequest(post.getTitle(), post.getContent());

        model.addAttribute("post", post);
        model.addAttribute("postRequest", postRequest);

        return "posts/edit";
    }

    @PostMapping("/{id}/edit")
    public String update(@PathVariable Long id,
                         @Valid @ModelAttribute PostRequest postRequest,
                         BindingResult bindingResult,
                         Authentication authentication,
                         Model model) {
        if (bindingResult.hasErrors()) {
            Post post = postService.findById(id);
            model.addAttribute("post", post);
            return "posts/edit";
        }

        String username = authentication.getName();
        postService.update(id, postRequest, username);

        return "redirect:/posts/" + id;
    }

    @PostMapping("/{id}/delete")
    public String delete(@PathVariable Long id,
                         Authentication authentication) {
        String username = authentication.getName();
        postService.delete(id, username);

        return "redirect:/posts";
    }

    private String getLoginUsername(Authentication authentication) {
        if (authentication == null || !authentication.isAuthenticated()) {
            return null;
        }

        if ("anonymousUser".equals(authentication.getName())) {
            return null;
        }

        return authentication.getName();
    }

    private boolean isAdmin(Authentication authentication) {
        if (authentication == null || !authentication.isAuthenticated()) {
            return false;
        }

        return authentication.getAuthorities().stream()
                .anyMatch(authority -> authority.getAuthority().equals("ROLE_ADMIN"));
    }

    private boolean canManage(Post post, String loginUsername, Authentication authentication) {
        if (loginUsername == null) {
            return false;
        }

        return post.isWrittenBy(loginUsername) || isAdmin(authentication);
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/controller/PostController.java


In [20]:
%%writefile ~/springBoot/security-jpa-board/src/main/java/com/example/board/controller/GlobalExceptionHandler.java

package com.example.board.controller;

import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.ControllerAdvice;
import org.springframework.web.bind.annotation.ExceptionHandler;

@ControllerAdvice
public class GlobalExceptionHandler {

    @ExceptionHandler(IllegalArgumentException.class)
    public String handleIllegalArgumentException(IllegalArgumentException e, Model model) {
        model.addAttribute("message", e.getMessage());
        return "error";
    }
}

Writing /root/springBoot/security-jpa-board/src/main/java/com/example/board/controller/GlobalExceptionHandler.java


In [21]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/static/css/style.css

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Noto Sans KR", Arial, sans-serif;
    background: #f5f6f8;
    color: #222;
}

.container {
    max-width: 920px;
    margin: 40px auto;
    padding: 0 20px;
}

.header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 20px;
}

.card {
    background: #fff;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 24px;
    box-shadow: 0 8px 24px rgba(0, 0, 0, 0.04);
}

.nav {
    display: flex;
    gap: 10px;
    align-items: center;
}

a {
    color: #2563eb;
    text-decoration: none;
}

a:hover {
    text-decoration: underline;
}

.btn {
    display: inline-block;
    border: none;
    border-radius: 10px;
    padding: 10px 14px;
    background: #111827;
    color: #fff;
    cursor: pointer;
    font-size: 14px;
    text-decoration: none;
}

.btn:hover {
    text-decoration: none;
    background: #374151;
}

.btn-secondary {
    background: #e5e7eb;
    color: #111827;
}

.btn-danger {
    background: #dc2626;
}

.btn-row {
    display: flex;
    gap: 8px;
    margin-top: 16px;
}

input,
textarea,
select {
    width: 100%;
    border: 1px solid #d1d5db;
    border-radius: 10px;
    padding: 12px;
    font-size: 15px;
    margin-top: 6px;
    background: #fff;
    color: #222;
}

select {
    cursor: pointer;
}

textarea {
    min-height: 240px;
    resize: vertical;
}

label {
    display: block;
    margin-top: 16px;
    font-weight: 600;
}

.error {
    color: #dc2626;
    font-size: 14px;
    margin-top: 6px;
}

.post-list {
    display: grid;
    gap: 12px;
}

.post-item {
    background: #fff;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 18px;
}

.post-title {
    font-size: 20px;
    font-weight: 700;
    margin: 0 0 8px;
}

.meta {
    color: #6b7280;
    font-size: 14px;
}

.content {
    white-space: pre-wrap;
    line-height: 1.7;
    margin-top: 20px;
}

Writing /root/springBoot/security-jpa-board/src/main/resources/static/css/style.css


In [22]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/templates/error.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>오류</title>
    <link rel="stylesheet" href="/css/style.css">
</head>
<body>
<div class="container">
    <h1>오류가 발생했습니다</h1>

    <div class="card">
        <p th:text="${message}">오류 메시지</p>
        <a class="btn" href="/posts">게시글 목록으로</a>
    </div>
</div>
</body>
</html>

Writing /root/springBoot/security-jpa-board/src/main/resources/templates/error.html


In [23]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/templates/auth/login.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>로그인</title>
    <link rel="stylesheet" href="/css/style.css">
</head>
<body>
<div class="container">
    <div class="header">
        <h1>로그인</h1>
        <div class="nav">
            <a href="/posts">게시글 목록</a>
            <a href="/auth/signup">회원가입</a>
        </div>
    </div>

    <div class="card">
        <form method="post" action="/login">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <label for="username">아이디</label>
            <input type="text" id="username" name="username" placeholder="아이디 입력">

            <label for="password">비밀번호</label>
            <input type="password" id="password" name="password" placeholder="비밀번호 입력">

            <div th:if="${param.error}" class="error">
                아이디 또는 비밀번호가 올바르지 않습니다.
            </div>

            <div th:if="${param.logout}" class="error">
                로그아웃되었습니다.
            </div>

            <div class="btn-row">
                <button class="btn" type="submit">로그인</button>
                <a class="btn btn-secondary" href="/auth/signup">회원가입</a>
            </div>
        </form>
    </div>
</div>
</body>
</html>

Writing /root/springBoot/security-jpa-board/src/main/resources/templates/auth/login.html


In [24]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/templates/auth/signup.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>회원가입</title>
    <link rel="stylesheet" href="/css/style.css">
</head>
<body>
<div class="container">
    <div class="header">
        <h1>회원가입</h1>
        <div class="nav">
            <a href="/posts">게시글 목록</a>
            <a href="/login">로그인</a>
        </div>
    </div>

    <div class="card">
        <form method="post" action="/auth/signup" th:object="${signupRequest}">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <label for="username">아이디</label>
            <input type="text" id="username" th:field="*{username}" placeholder="아이디 입력">
            <div class="error" th:if="${#fields.hasErrors('username')}" th:errors="*{username}"></div>

            <label for="password">비밀번호</label>
            <input type="password" id="password" th:field="*{password}" placeholder="비밀번호 입력">
            <div class="error" th:if="${#fields.hasErrors('password')}" th:errors="*{password}"></div>

            <label for="displayName">이름</label>
            <input type="text" id="displayName" th:field="*{displayName}" placeholder="이름 입력">
            <div class="error" th:if="${#fields.hasErrors('displayName')}" th:errors="*{displayName}"></div>

            <label for="role">권한</label>
            <select id="role" th:field="*{role}">
                <option value="USER">일반 사용자</option>
                <option value="ADMIN">관리자</option>
            </select>
            <div class="error" th:if="${#fields.hasErrors('role')}" th:errors="*{role}"></div>

            <div class="error" th:if="${signupError}" th:text="${signupError}"></div>

            <div class="btn-row">
                <button class="btn" type="submit">가입하기</button>
                <a class="btn btn-secondary" href="/login">로그인</a>
            </div>
        </form>
    </div>
</div>
</body>
</html>

Writing /root/springBoot/security-jpa-board/src/main/resources/templates/auth/signup.html


In [25]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/templates/posts/list.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>게시글 목록</title>
    <link rel="stylesheet" href="/css/style.css">
</head>
<body>
<div class="container">
    <div class="header">
        <h1>게시글 목록</h1>

        <div class="nav">
            <span th:if="${loginUsername != null}" th:text="${loginUsername + '님'}"></span>
            <span th:if="${isAdmin}" class="meta">관리자</span>

            <a th:if="${loginUsername == null}" href="/login">로그인</a>
            <a th:if="${loginUsername == null}" href="/auth/signup">회원가입</a>

            <a th:if="${loginUsername != null}" class="btn" href="/posts/new">글쓰기</a>

            <form th:if="${loginUsername != null}" method="post" action="/logout" style="display:inline;">
                <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
                <button class="btn btn-secondary" type="submit">로그아웃</button>
            </form>
        </div>
    </div>

    <div class="post-list">
        <div th:if="${#lists.isEmpty(posts)}" class="card">
            등록된 게시글이 없습니다.
        </div>

        <div th:each="post : ${posts}" class="post-item">
            <h2 class="post-title">
                <a th:href="@{/posts/{id}(id=${post.id})}" th:text="${post.title}">게시글 제목</a>
            </h2>
            <div class="meta">
                작성자:
                <span th:text="${post.writer.displayName}">작성자</span>
                |
                작성일:
                <span th:text="${#temporals.format(post.createdAt, 'yyyy-MM-dd HH:mm')}">작성일</span>
            </div>
        </div>
    </div>
</div>
</body>
</html>

Writing /root/springBoot/security-jpa-board/src/main/resources/templates/posts/list.html


In [26]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/templates/posts/detail.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>게시글 상세</title>
    <link rel="stylesheet" href="/css/style.css">
</head>
<body>
<div class="container">
    <div class="header">
        <h1>게시글 상세</h1>
        <div class="nav">
            <span th:if="${isAdmin}" class="meta">관리자</span>

            <a href="/posts">목록</a>
            <a th:if="${loginUsername == null}" href="/login">로그인</a>

            <form th:if="${loginUsername != null}" method="post" action="/logout" style="display:inline;">
                <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
                <button class="btn btn-secondary" type="submit">로그아웃</button>
            </form>
        </div>
    </div>

    <div class="card">
        <h2 th:text="${post.title}">게시글 제목</h2>

        <div class="meta">
            작성자:
            <span th:text="${post.writer.displayName}">작성자</span>
            |
            작성일:
            <span th:text="${#temporals.format(post.createdAt, 'yyyy-MM-dd HH:mm')}">작성일</span>
            <span th:if="${post.updatedAt != null}">
                |
                수정일:
                <span th:text="${#temporals.format(post.updatedAt, 'yyyy-MM-dd HH:mm')}">수정일</span>
            </span>
        </div>

        <div class="content" th:text="${post.content}">
            게시글 내용
        </div>

        <div class="btn-row">
            <a class="btn btn-secondary" href="/posts">목록</a>

            <a th:if="${canManage}"
               class="btn"
               th:href="@{/posts/{id}/edit(id=${post.id})}">
                수정
            </a>

            <form th:if="${canManage}"
                  method="post"
                  th:action="@{/posts/{id}/delete(id=${post.id})}"
                  style="display:inline;">
                <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
                <button class="btn btn-danger" type="submit">삭제</button>
            </form>
        </div>
    </div>
</div>
</body>
</html>

Writing /root/springBoot/security-jpa-board/src/main/resources/templates/posts/detail.html


In [27]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/templates/posts/form.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>게시글 등록</title>
    <link rel="stylesheet" href="/css/style.css">
</head>
<body>
<div class="container">
    <div class="header">
        <h1>게시글 등록</h1>
        <div class="nav">
            <a href="/posts">목록</a>
        </div>
    </div>

    <div class="card">
        <form method="post" action="/posts" th:object="${postRequest}">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <label for="title">제목</label>
            <input type="text" id="title" th:field="*{title}" placeholder="제목 입력">
            <div class="error" th:if="${#fields.hasErrors('title')}" th:errors="*{title}"></div>

            <label for="content">내용</label>
            <textarea id="content" th:field="*{content}" placeholder="내용 입력"></textarea>
            <div class="error" th:if="${#fields.hasErrors('content')}" th:errors="*{content}"></div>

            <div class="btn-row">
                <button class="btn" type="submit">등록</button>
                <a class="btn btn-secondary" href="/posts">취소</a>
            </div>
        </form>
    </div>
</div>
</body>
</html>

Writing /root/springBoot/security-jpa-board/src/main/resources/templates/posts/form.html


In [28]:
%%writefile ~/springBoot/security-jpa-board/src/main/resources/templates/posts/edit.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>게시글 수정</title>
    <link rel="stylesheet" href="/css/style.css">
</head>
<body>
<div class="container">
    <div class="header">
        <h1>게시글 수정</h1>
        <div class="nav">
            <a th:href="@{/posts/{id}(id=${post.id})}">상세</a>
            <a href="/posts">목록</a>
        </div>
    </div>

    <div class="card">
        <form method="post" th:action="@{/posts/{id}/edit(id=${post.id})}" th:object="${postRequest}">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <label for="title">제목</label>
            <input type="text" id="title" th:field="*{title}" placeholder="제목 입력">
            <div class="error" th:if="${#fields.hasErrors('title')}" th:errors="*{title}"></div>

            <label for="content">내용</label>
            <textarea id="content" th:field="*{content}" placeholder="내용 입력"></textarea>
            <div class="error" th:if="${#fields.hasErrors('content')}" th:errors="*{content}"></div>

            <div class="btn-row">
                <button class="btn" type="submit">수정</button>
                <a class="btn btn-secondary" th:href="@{/posts/{id}(id=${post.id})}">취소</a>
            </div>
        </form>
    </div>
</div>
</body>
</html>

Writing /root/springBoot/security-jpa-board/src/main/resources/templates/posts/edit.html
